# 📱 手机产线尺寸相关性分析（教学项目）

> **目标**：用一套 500×200 的"手机产线组装前后尺寸"数据集，系统讲解**数据相关性分析**的全部主流方法，
> 重点解决你关心的两个问题：
> 1. 组装前的零件尺寸与组装后的整机尺寸之间有怎样的关联？
> 2. 哪些尺寸是导致**不良品（0/1）** 的关键 X 因子？

**数据全部在本 Notebook 内用种子随机数生成，可复现，无需任何外部文件。**

---

## 📋 章节地图

| 章 | 主题 | 核心方法 |
|---|---|---|
| 0 | 项目背景与数据生成 | 潜在工艺因子构造 |
| 1 | 环境准备与数据加载 | pandas |
| 2 | 探索性数据分析 (EDA) | 分布、箱线图 |
| 3 | 连续变量间相关性 | Pearson / Spearman / Kendall + 热力图 + 层次聚类 |
| 4 | ⭐ 0/1 不良品 vs 尺寸的关联 | 点二列相关 / t 检验 / Mann-Whitney / 单变量逻辑回归 + ROC |
| 5 | 多元逻辑回归（不良品 X 因子建模） | statsmodels + VIF 共线性诊断 |
| 6 | 降维：寻找潜在工艺因子 | PCA + 探索性因子分析 (EFA) |
| 7 | 机器学习对比：随机森林特征重要性 | sklearn + SHAP 思路 |
| 8 | 综合结论与实战练习题 | — |

> 🎯 **教学闭环**：数据里故意"埋"了 6 个潜在工艺因子，你能通过分析逐步"恢复"它们，并在最后与真相对照。


---
## 第 0 章 · 项目背景与数据生成

### 0.1 业务背景

手机组装产线上，每台手机在**组装前**会测量大量零件尺寸（屏幕玻璃厚度、中框各点位、电池仓、FPC 宽度、摄像头模组高度、螺丝孔坐标……），
**组装后**又会测量整机尺寸（整机厚/长/宽、屏幕-中框间隙、贴合平整度、按键行程……）。典型一条产线一次工单会有**几百个测量点**。

质量工程师关心：

- 这些尺寸之间**谁和谁相关**？（哪个零件尺寸波动会牵连哪个整机尺寸？）
- 哪些尺寸是**不良品的预测因子**？（提前在组装前就能预警？）
- 能不能把上百个高度相关的尺寸**压缩成少数几个"工艺因子"**，便于归因？

本 Notebook 就用一套模拟数据，把这些分析方法**完整走一遍**。

### 0.2 数据"真相"（先剧透，方便你后面对照分析结果）

500 台手机 × 200 列：
- `X_pre_001` ~ `X_pre_100`（100 列）：组装前零件尺寸
- `X_post_001` ~ `X_post_099`（99 列）：组装后整机尺寸
- `label`：不良品 `0/1`

所有尺寸都由 **6 个不可观测的潜在工艺因子** F1~F6 驱动：

| 因子 | 工艺含义 | 对不良的影响 |
|---|---|---|
| **F1** | 点胶密封 | **强相关** |
| **F2** | 屏幕贴合 | **强相关** |
| **F3** | 中框冲压 | 中等相关 |
| F4 | 电池装配 | 弱相关 |
| F5 | 螺丝锁附扭矩 | 噪声（无关） |
| F6 | 环境温湿度 | 噪声（无关） |

- 每个尺寸 = 6 因子的线性组合（不同尺寸有不同载荷）+ 噪声 → 自然产生**多重共线性**
- 不良概率 = `sigmoid(β0 + 2.0·F1 + 1.8·F2 + 0.6·F3)` → 让逻辑回归 / 树模型都能挖出来

> 接下来我们先**生成**这套数据，再假装"什么也不知道"地从第 1 章开始分析。

In [ ]:
# 0.3 数据生成（种子固定，可复现）
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

N = 500          # 样本数（500 台手机）
N_PRE = 100      # 组装前零件尺寸列数
N_POST = 99      # 组装后整机尺寸列数
N_TOTAL = N_PRE + N_POST  # 199 个尺寸列 + 1 个 label = 200 列
N_FACTORS = 6    # 潜在工艺因子数

# ---- 第 1 步：生成 6 个潜在工艺因子（标准正态）----
F = rng.standard_normal((N, N_FACTORS))
factor_names = ['F1_点胶密封', 'F2_屏幕贴合', 'F3_中框冲压', 'F4_电池装配', 'F5_螺丝扭矩', 'F6_环境温湿']
print("潜在因子矩阵 F 的形状：", F.shape)

# ---- 第 2 步：生成不良标签（由 F1/F2/F3 决定，F4~F6 与不良无关）----
# 不良率约 8%~12%
# 截距 -3.7 用于抵消潜因子方差对 sigmoid 期望的抬升，使不良率落在 ~10-12%
z = -3.7 + 2.0 * F[:, 0] + 1.8 * F[:, 1] + 0.6 * F[:, 2]   # 真实的对数几率
p_defect = 1 / (1 + np.exp(-z))                             # sigmoid
label = (rng.random(N) < p_defect).astype(int)
print(f"不良率：{label.mean():.1%}（{label.sum()}/{N}）")

In [ ]:
# ---- 第 3 步：构造"载荷矩阵" —— 每个尺寸列对 6 个因子的依赖程度 ----
# 设计思路：让不同尺寸列"偏好"不同的因子，制造出可识别的结构。
# 比如前若干列主要由 F1 驱动，中间几列主要由 F2 驱动……
def build_loadings(n_cols, n_factors, rng, noise_scale=0.3):
    """每个尺寸列随机选 1~2 个主因子，其余因子小载荷，再加噪声。"""
    W = np.zeros((n_cols, n_factors))
    for j in range(n_cols):
        # 主因子：1~2 个
        n_main = rng.choice([1, 2], p=[0.6, 0.4])
        main_idx = rng.choice(n_factors, size=n_main, replace=False)
        for mi in main_idx:
            W[j, mi] = rng.uniform(0.6, 1.0) * rng.choice([-1, 1])
        # 次因子：小载荷
        for k in range(n_factors):
            if k not in main_idx and rng.random() < 0.3:
                W[j, k] = rng.uniform(-0.25, 0.25)
    return W

W_pre = build_loadings(N_PRE, N_FACTORS, rng)
W_post = build_loadings(N_POST, N_FACTORS, rng)
print("组装前载荷矩阵形状：", W_pre.shape)
print("组装后载荷矩阵形状：", W_post.shape)

# ---- 第 4 步：尺寸 = F @ W^T + 噪声 ----
noise_pre = rng.standard_normal((N, N_PRE)) * 0.5
noise_post = rng.standard_normal((N, N_POST)) * 0.5
X_pre = F @ W_pre.T + noise_pre
X_post = F @ W_post.T + noise_post

# 拼成完整 DataFrame
col_pre = [f'X_pre_{i:03d}' for i in range(1, N_PRE + 1)]
col_post = [f'X_post_{i:03d}' for i in range(1, N_POST + 1)]
df = pd.DataFrame(np.hstack([X_pre, X_post]), columns=col_pre + col_post)
df['label'] = label

print("\n数据集形状：", df.shape)
df.head()

In [ ]:
# 0.4 把数据存到全局，后续章节复用；同时导出一份 CSV 方便外部工具打开（可选）
import os
os.makedirs('output', exist_ok=True)
df.to_csv('output/phone_assembly_dataset.csv', index=False, encoding='utf-8-sig')
print("数据已保存到 output/phone_assembly_dataset.csv")
print("\n字段列名示例（前 3 + 后 3）：")
print(list(df.columns[:3]), '...', list(df.columns[-3:]))

---
## 第 1 章 · 环境准备与数据加载

> 本章把后续要用到的库一次性导入，并对数据做**最基本的体检**：形状、数据类型、缺失值、明显异常值。
> 这是任何数据分析项目的第一步——**先看清数据长什么样，再谈方法**。

In [ ]:
# 1.1 导入全部依赖（如果某些库缺失，请 pip install -r requirements.txt）
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import cross_val_score

# 中文字体配置（Windows 常用黑体；如显示方块请改成你机器上有的中文字体）
mpl.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
mpl.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

print("✅ 依赖加载完成")
print("pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
# 1.2 如果跳过了第 0 章的生成，可以从 CSV 重新加载（否则直接用内存里的 df）
# df = pd.read_csv('output/phone_assembly_dataset.csv')

# 数据体检：形状、列、数据类型
print("数据形状（行×列）：", df.shape)
print("\n数据类型分布：")
print(df.dtypes.value_counts())
print("\n前 5 行预览：")
df.head()

In [ ]:
# 1.3 缺失值检查
missing = df.isnull().sum()
print("缺失值总数：", missing.sum())
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("✅ 无缺失值（本模拟数据干净；真实产线数据通常会有 MES 漏采）")

In [ ]:
# 1.4 描述性统计 + 明显异常值初筛
desc = df.describe()
desc

In [ ]:
# 1.5 用 Z 分位看哪些尺寸有极端值（|z|>5 视为可疑，仅作标记，不删除）
feature_cols = [c for c in df.columns if c != 'label']
z = (df[feature_cols] - df[feature_cols].mean()) / df[feature_cols].std(ddof=0)
extreme_counts = (z.abs() > 5).sum()
extreme_cols = extreme_counts[extreme_counts > 0].sort_values(ascending=False)
print(f"存在 |z|>5 极端值的尺寸数：{len(extreme_cols)}")
print(extreme_cols.head(10) if len(extreme_cols) > 0 else "无")
print("\n💡 教学提示：异常值处理是质量数据分析的大学问，本案例数据干净，先跳过；",
      "真实数据需结合测量原理判断是'真异常'还是'测量噪声'。")

---
## 第 2 章 · 探索性数据分析 (EDA)

> **EDA（Exploratory Data Analysis）** 的目标是用图形快速建立对数据的"直觉"：
> 尺寸大致服从什么分布？良品和不良品的尺寸分布有差异吗？哪些尺寸波动大？
>
> EDA 不下结论，只为后续建模**指方向**。

In [ ]:
# 2.1 不良率总览
ax = df['label'].value_counts().plot(kind='bar', color=['#4CAF50', '#E53935'])
plt.title(f'良品 vs 不良品（不良率 {df["label"].mean():.1%}）')
plt.xticks([0, 1], ['良品 (0)', '不良品 (1)'], rotation=0)
plt.ylabel('数量')
for i, v in enumerate(df['label'].value_counts().reindex([0, 1])):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.show()

In [ ]:
# 2.2 尺寸波动大小排行（标准差 Top 20）
# 哪些尺寸波动最大？波动大的尺寸往往是工艺控制的重点
std_top = df[feature_cols].std().sort_values(ascending=False).head(20)
plt.figure(figsize=(10, 5))
std_top.plot(kind='bar', color=sns.color_palette('viridis', 20))
plt.title('波动最大的 20 个尺寸（标准差）')
plt.ylabel('标准差')
plt.tight_layout()
plt.show()
print("💡 提示：标准差大 = 该尺寸一致性差，是 SPC（统计过程控制）优先监控对象。")

In [ ]:
# 2.3 良品 vs 不良品 的尺寸分布对比（随机抽 9 个尺寸看箱线图）
# 如果不良品分布明显偏移，说明该尺寸可能与不良有关
sample_cols = np.random.default_rng(7).choice(feature_cols, 9, replace=False)
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, col in zip(axes.flat, sample_cols):
    sns.boxplot(data=df, x='label', y=col, ax=ax, palette=['#4CAF50', '#E53935'])
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('')
    ax.set_xticklabels(['良', '不良'])
plt.suptitle('随机抽样 9 个尺寸：良品 vs 不良品 分布对比', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()
print("💡 观察：哪些子图的两个箱体明显错开？那些尺寸可能就是不良的相关因子（第 4 章会量化）。")

In [ ]:
# 2.4 直方图看整体分布形态（抽 6 个尺寸）
sample6 = np.random.default_rng(11).choice(feature_cols, 6, replace=False)
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, col in zip(axes.flat, sample6):
    sns.histplot(df[col], kde=True, ax=ax, color='#1976D2', bins=30)
    ax.set_title(col, fontsize=9)
plt.suptitle('部分尺寸的分布形态', y=1.01)
plt.tight_layout()
plt.show()
print("💡 教学提示：大部分工业尺寸应近似正态。若出现双峰/长尾，可能是多批次混合或测量异常。")

---
## 第 3 章 · 连续变量间的相关性分析

> **核心问题**：199 个尺寸两两之间，谁和谁相关？
>
> 这是相关性分析最经典的场景。三种相关系数要分清：

| 系数 | 衡量什么 | 取值范围 | 适用场景 |
|---|---|---|---|
| **Pearson** | **线性**相关 | [-1, 1] | 数据近似正态、关系是直线 |
| **Spearman** | **单调**相关（基于秩） | [-1, 1] | 非线性单调关系、有异常值 |
| **Kendall** | 一致性（基于秩对） | [-1, 1] | 小样本、要更稳健 |

工业尺寸数据通常近似正态，**默认先用 Pearson**，再用 Spearman 做交叉验证。

In [ ]:
# 3.1 计算 Pearson 相关矩阵（199×199）
corr_pearson = df[feature_cols].corr(method='pearson')
print("Pearson 相关矩阵形状：", corr_pearson.shape)
print("\n正相关最强的 10 对尺寸（不含自身）：")
# 取上三角，避免重复
mask = np.triu(np.ones_like(corr_pearson, dtype=bool), k=1)
pairs = corr_pearson.where(mask).stack().sort_values(ascending=False)
print(pairs.head(10).to_string())
print("\n负相关最强的 10 对尺寸：")
print(pairs.tail(10)[::-1].to_string())

In [ ]:
# 3.2 相关性热力图（199×199 全景）
# 由于变量多，用层次聚类重排，让"成团"的相关变量聚到一起
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

# 用 1 - |r| 作为距离做层次聚类
d = 1 - corr_pearson.abs().values
np.fill_diagonal(d, 0)
d = (d + d.T) / 2
condensed = squareform(d, checks=False)
Z = linkage(condensed, method='average')
order = leaves_list(Z)
ordered_corr = corr_pearson.iloc[order, order]

plt.figure(figsize=(11, 9))
sns.heatmap(ordered_corr, cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True,
            cbar_kws={'label': 'Pearson r', 'shrink': 0.7},
            xticklabels=False, yticklabels=False)
plt.title('199 个尺寸的 Pearson 相关性热力图（按层次聚类重排）')
plt.tight_layout()
plt.show()
print("💡 观察：对角线上成块的'红/蓝方阵' = 高度相关的尺寸组。")
print("   这些块就是后续 PCA/因子分析能压缩出来的'潜在工艺因子'。")

In [ ]:
# 3.3 组装前 vs 组装后 的跨阶段相关性
# 这是本项目的核心业务问题之一：哪些"组装前尺寸"和"组装后尺寸"强相关？
corr_cross = corr_pearson.loc[col_pre, col_post]
print("跨阶段相关矩阵形状：", corr_cross.shape)

# 找最强的 15 对"前→后"关联
cross_pairs = corr_cross.stack().sort_values(key=abs, ascending=False).head(15)
print("\n|相关性| 最强的 15 对（组装前 → 组装后）：")
print(cross_pairs.to_string())

In [ ]:
# 3.4 跨阶段相关性热力图（前 100 × 后 99）
plt.figure(figsize=(10, 8))
sns.heatmap(corr_cross, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            cbar_kws={'label': 'Pearson r', 'shrink': 0.7},
            xticklabels=8, yticklabels=8)
plt.title('组装前尺寸 × 组装后尺寸 相关性热力图')
plt.xlabel('组装后尺寸（X_post）')
plt.ylabel('组装前尺寸（X_pre）')
plt.tight_layout()
plt.show()
print("💡 业务解读：图中颜色鲜明的'行/列'代表某个零件尺寸强烈影响多个整机尺寸，")
print("   或某个整机尺寸被多个零件尺寸共同决定——这些是工艺工程师要重点关注的'耦合点'。")

In [ ]:
# 3.5 Pearson vs Spearman 交叉验证（挑几个强相关的对子看是否一致）
top6 = pairs.head(6).index.tolist()
print("Pearson 与 Spearman 对比（验证关系是否稳定、是否非线性）：")
print(f"{'尺寸A':<14}{'尺寸B':<14}{'Pearson':>10}{'Spearman':>10}{'差异':>8}")
for a, b in top6:
    rp = df[a].corr(df[b], method='pearson')
    rs = df[a].corr(df[b], method='spearman')
    print(f"{a:<14}{b:<14}{rp:>10.3f}{rs:>10.3f}{abs(rp-rs):>8.3f}")
print("\n💡 解读：两者接近 → 线性关系；差异大 → 可能存在非线性单调关系，需用 Spearman 更稳健。")

In [ ]:
# 3.6 散点图抽样：直观验证几对强相关尺寸
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (a, b) in zip(axes.flat, top6):
    ax.scatter(df[a], df[b], s=8, alpha=0.5, c=df['label'].map({0:'#4CAF50', 1:'#E53935'}))
    ax.set_xlabel(a, fontsize=8); ax.set_ylabel(b, fontsize=8)
    ax.set_title(f"r = {df[a].corr(df[b]):.2f}", fontsize=9)
plt.suptitle('强相关尺寸对的散点图（绿=良品，红=不良品）', y=1.01)
plt.tight_layout()
plt.show()
print("💡 进阶观察：在散点图里，红色点（不良品）是否集中在某个区域？这就是'分群异常'的信号。")

---
## 第 4 章 · ⭐ 0/1 不良品标签与尺寸的关联分析（核心案例）

> **这是本项目的重头戏**：label 是 0/1 二分类，尺寸是连续变量。
> "连续变量 vs 二分类变量"的相关性，**不能直接用 Pearson**！要用专门的方法：
>
> | 方法 | 原理 | 输出 |
> |---|---|---|
> | **点二列相关** (point-biserial) | 专门衡量"二分类 vs 连续"的关联 | r ∈ [-1,1] |
> | **两样本 t 检验** | 良品组与不良品组的尺寸均值是否显著不同 | p 值 |
> | **Mann-Whitney U** | 非参数版的两组比较（不要正态假设） | p 值 |
> | **单变量逻辑回归 + AUC** | 该尺寸单独预测不良的能力 | AUC ∈ [0.5,1] |
>
> 一句话总结：**点二列相关 + t 检验 + AUC 三件套**，是工业质量里"找关键 X 因子"的标准流程。

In [ ]:
# 4.1 点二列相关：扫描全部 199 个尺寸
# scipy.stats.pointbiserialr 专门处理"0/1 vs 连续"
from scipy.stats import pointbiserialr

results = []
for col in feature_cols:
    r, p = pointbiserialr(df['label'], df[col])
    results.append({'尺寸': col, '点二列r': r, 'p值': p,
                    '|r|': abs(r), '是否组装后': col.startswith('X_post')})
pb_df = pd.DataFrame(results).sort_values('|r|', ascending=False).reset_index(drop=True)
pb_df['排名'] = pb_df.index + 1
print("点二列相关 Top 20 关键尺寸：")
pb_df[['排名', '尺寸', '点二列r', 'p值', '是否组装后']].head(20)

In [ ]:
# 4.2 Top 关键尺寸可视化：条形图
top20 = pb_df.head(20).copy()
colors = ['#E53935' if x else '#1976D2' for x in top20['是否组装后']]
plt.figure(figsize=(10, 6))
plt.barh(top20['尺寸'][::-1], top20['点二列r'][::-1], color=colors[::-1])
plt.xlabel('点二列相关系数 r（正=r越大越不良，负=r越小越不良）')
plt.title('与不良品关联最强的 Top 20 尺寸')
from matplotlib.patches import Patch
plt.legend(handles=[Patch(color='#1976D2', label='组装前尺寸 X_pre'),
                    Patch(color='#E53935', label='组装后尺寸 X_post')],
           loc='lower right')
plt.tight_layout()
plt.show()
print("💡 关键观察：红色（组装后）多还是蓝色（组装前）多？")
print("   如果组装前尺寸上榜多 → 可以在组装前就预警不良（更有价值）。")

In [ ]:
# 4.3 两样本 t 检验 + Mann-Whitney U（非参数对照）
# 对 Top 20 尺寸，分别做参数(t) 和非参数(MW) 检验，交叉验证
test_rows = []
for _, row in pb_df.head(20).iterrows():
    col = row['尺寸']
    g0 = df.loc[df['label'] == 0, col]
    g1 = df.loc[df['label'] == 1, col]
    t_stat, t_p = stats.ttest_ind(g0, g1, equal_var=False)  # Welch t
    u_stat, u_p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
    test_rows.append({'尺寸': col, '点二列r': row['点二列r'],
                      't统计量': t_stat, 't_p值': t_p,
                      'MW_p值': u_p,
                      '良品均值': g0.mean(), '不良均值': g1.mean()})
test_df = pd.DataFrame(test_rows)
test_df['显著(t<0.05)'] = test_df['t_p值'] < 0.05
test_df

In [ ]:
# 4.4 多重检验校正（重要！）
# 我们做了 199 次假设检验，单纯 p<0.05 会有大量假阳性。
# 用 Bonferroni / Benjamini-Hochberg(FDR) 校正。
from statsmodels.stats.multitest import multipletests

# 对全部 199 个尺寸的 p 值做 FDR 校正
_, p_bh, _, _ = multipletests(pb_df['p值'], method='fdr_bh')
pb_df['FDR校正p值'] = p_bh
pb_df['FDR显著'] = p_bh < 0.05
n_sig = pb_df['FDR显著'].sum()
print(f"经 Benjamini-Hochberg FDR 校正后，仍有 {n_sig} 个尺寸与不良显著相关（q<0.05）")
print("\n💡 教学要点：做大量并行检验时，必须做多重校正，否则'显著'结果里一半是假的。")
print("   这一步在真实产线分析中常被忽略，导致工程师去优化一个其实无关的尺寸。")

In [ ]:
# 4.5 单变量逻辑回归 + ROC/AUC：量化每个尺寸的"预测力"
# AUC 是衡量二分类预测力的标准指标：0.5=瞎猜，1=完美，>0.7 可用，>0.8 强
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc

X_all = df[feature_cols].values
y = df['label'].values

auc_results = []
for col in feature_cols:
    x = df[[col]].values
    # 用交叉验证更稳健，这里简化为全样本拟合（教学用）
    clf = LogisticRegression(max_iter=1000)
    clf.fit(x, y)
    proba = clf.predict_proba(x)[:, 1]
    fpr, tpr, _ = roc_curve(y, proba)
    auc_results.append({'尺寸': col, 'AUC': auc(fpr, tpr)})
auc_df = pd.DataFrame(auc_results).sort_values('AUC', ascending=False).reset_index(drop=True)
print("单变量预测不良的 AUC Top 15：")
auc_df.head(15)

In [ ]:
# 4.6 Top 尺寸的 ROC 曲线
top_auc = auc_df.head(6)['尺寸'].tolist()
plt.figure(figsize=(8, 6))
for col in top_auc:
    x = df[[col]].values
    clf = LogisticRegression(max_iter=1000).fit(x, y)
    proba = clf.predict_proba(x)[:, 1]
    fpr, tpr, _ = roc_curve(y, proba)
    plt.plot(fpr, tpr, label=f'{col} (AUC={auc(fpr, tpr):.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='随机 (AUC=0.5)')
plt.xlabel('假阳率 FPR')
plt.ylabel('真阳率 TPR')
plt.title('Top 6 尺寸预测不良品的 ROC 曲线')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()
print("💡 解读：AUC 越接近 1，该尺寸单独就能越好地区分良/不良品。")

In [ ]:
# 4.7 三件套汇总：把点二列r、t检验p、AUC 合并成一张"风险尺寸排行榜"
risk = pb_df[['尺寸', '点二列r', '|r|', 'FDR显著']].merge(
    auc_df, on='尺寸').merge(
    test_df[['尺寸', 't_p值', 'MW_p值', '良品均值', '不良均值']], on='尺寸', how='left')
risk = risk.sort_values('AUC', ascending=False).reset_index(drop=True)
print("=" * 70)
print("🏆 不良品关键 X 因子排行榜（综合点二列r / t检验 / AUC）")
print("=" * 70)
risk.head(15)

### 📌 第 4 章小结

通过 **点二列相关 + t/MW 检验（含 FDR 校正）+ 单变量逻辑回归 AUC** 三件套，
我们从 199 个尺寸里**客观地**筛出了一批与不良品强相关的"风险尺寸"。

**但要警惕**：单变量分析只看"单个尺寸 vs 不良"，**忽略了尺寸之间的相互关联**。
比如两个高度相关的尺寸可能都"显著"，但其实背后是同一个工艺因子。
→ 这正是第 5 章（多元逻辑回归 + 共线性诊断）和第 6 章（降维找因子）要解决的。

---
## 第 5 章 · 多元逻辑回归：不良品的 X 因子建模 + 多重共线性诊断

> 第 4 章是"单变量"视角，本章是"**多变量**"视角：把所有（或一批）尺寸**同时**放进逻辑回归，
> 看在**控制了其他尺寸后**，哪些尺寸对不良还有独立贡献。
>
> **核心痛点：多重共线性**。我们的 199 个尺寸高度相关（第 3 章热力图已看到），
> 直接全塞进逻辑回归会出问题：
> - 系数方差膨胀，p 值不可靠
> - 系数符号可能"反常"
>
> **诊断工具：VIF（方差膨胀因子）**。经验法则：VIF > 10 表示严重共线性。

In [ ]:
# 5.1 先用 VIF 诊断全部 199 个尺寸的共线性
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X = df[feature_cols].values
# 标准化（VIF 对尺度敏感）
X_scaled = StandardScaler().fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=feature_cols)
X_const = add_constant(X_df)

vif = pd.DataFrame()
vif['尺寸'] = feature_cols
vif['VIF'] = [variance_inflation_factor(X_const.values, i + 1) for i in range(len(feature_cols))]
vif = vif.sort_values('VIF', ascending=False).reset_index(drop=True)
print("VIF Top 20（>10 视为严重共线性）：")
print(vif.head(20).to_string(index=False))
n_high = (vif['VIF'] > 10).sum()
print(f"\nVIF>10 的尺寸数：{n_high} / {len(feature_cols)}")
print("💡 结论：果然高度共线！全变量直接进逻辑回归不可行 → 需要先降维或筛变量。")

In [ ]:
# 5.2 策略：用第 4 章筛出的 Top 风险尺寸 + 逐步剔除高 VIF
# 取 AUC 排名前 30 的尺寸作为候选（既覆盖信号又控制变量数）
top_candidates = risk.head(30)['尺寸'].tolist()

def reduce_vif(cols, max_vif=10.0):
    """迭代剔除 VIF 最高的变量，直到全部 VIF < max_vif。"""
    cols = list(cols)
    while True:
        Xs = add_constant(df[cols])
        vifs = pd.Series(
            [variance_inflation_factor(Xs.values, i + 1) for i in range(len(cols))],
            index=cols)
        if vifs.max() < max_vif or len(cols) <= 2:
            return cols, vifs
        worst = vifs.idxmax()
        cols.remove(worst)
    return cols, vifs

kept_cols, final_vifs = reduce_vif(top_candidates, max_vif=10.0)
print(f"VIF 筛选后保留 {len(kept_cols)} 个尺寸（共线性已控制）：")
print(final_vifs.sort_values(ascending=False).to_string())

In [ ]:
# 5.3 多元逻辑回归（statsmodels 给出完整统计报告）
import statsmodels.api as sm

X_model = df[kept_cols].copy()
X_model = add_constant(X_model)
logit = sm.Logit(df['label'], X_model)
result = logit.fit(disp=0, maxiter=200)
print(result.summary())

In [ ]:
# 5.4 解读：系数、Odds Ratio、显著性
coef_df = pd.DataFrame({
    '系数': result.params,
    '标准误': result.bse,
    'z值': result.tvalues,
    'p值': result.pvalues,
    'OR(优势比)': np.exp(result.params),
})
coef_df = coef_df.drop('const').sort_values('p值')
coef_df['显著(p<0.05)'] = coef_df['p值'] < 0.05
print("多元逻辑回归系数解读（按 p 值升序）：")
print(coef_df.to_string())
print("\n💡 解读要点：")
print("  - OR>1：该尺寸增大→不良几率升高；OR<1：相反")
print("  - p<0.05：在控制其他变量后，该尺寸对不良仍有显著独立贡献")

In [ ]:
# 5.5 多元模型的预测力（与第 4 章单变量 AUC 对比）
proba_full = result.predict(X_model)
fpr, tpr, _ = roc_curve(df['label'], proba_full)
auc_full = auc(fpr, tpr)
print(f"多元逻辑回归整体 AUC = {auc_full:.3f}")
print(f"对比：第 4 章最强单变量 AUC = {auc_df['AUC'].iloc[0]:.3f}（{auc_df['尺寸'].iloc[0]}）")
print("\n💡 解读：多元模型通常比单变量 AUC 高，但若高得不多，说明少数关键尺寸已抓住主要信号。")

### 📌 第 5 章小结

- 多元逻辑回归回答的是"**控制其他变量后，X 还重要吗？**"，比单变量更严谨
- 但本数据集**高度共线**，必须先用 VIF 筛变量，否则系数不可靠
- 真实产线建议：**先降维（第 6 章 PCA/因子分析）得到几个独立因子，再用因子做回归**，更稳健

---
## 第 6 章 · 降维：寻找潜在工艺因子（PCA + 探索性因子分析）

> 还记得第 0 章"剧透"的真相吗？**6 个潜在因子** 驱动了全部 199 个尺寸。
> 现在我们假装不知道，用 **PCA** 和 **探索性因子分析 (EFA)** 把它们"挖"出来。
>
> - **PCA**：找方差最大的方向，主成分是原变量的线性组合
> - **因子分析 (EFA)**：假设背后有不可观测的潜因子，更贴近"工艺因子"的物理直觉
>
> 两者都能把 199 维压到 ~6 维，且**结果可解释**。

In [ ]:
# 6.1 PCA：先标准化，再看解释方差比
X_scaled = StandardScaler().fit_transform(df[feature_cols])
pca = PCA()
pca.fit(X_scaled)
exp = pca.explained_variance_ratio_
cum = np.cumsum(exp)

# Scree plot：前若干主成分的方差贡献
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.bar(range(1, 11), exp[:10], color='#1976D2', alpha=0.7, label='单个')
plt.plot(range(1, 11), cum[:10], 'o-', color='#E53935', label='累计')
plt.xlabel('主成分'); plt.ylabel('方差占比')
plt.title('PCA Scree Plot（前 10 个主成分）')
plt.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='90%阈值')
plt.legend()
plt.subplot(1, 2, 2)
# 找累计解释 90% 需要多少个 PC
n_90 = np.argmax(cum >= 0.9) + 1
plt.plot(range(1, len(cum) + 1), cum, color='#1976D2')
plt.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=n_90, color='red', linestyle='--', alpha=0.7)
plt.xlabel('主成分数'); plt.ylabel('累计方差占比')
plt.title(f'累计解释方差（达到90%需 {n_90} 个主成分）')
plt.tight_layout()
plt.show()
print(f"💡 用 {n_90} 个主成分就能解释 90% 的尺寸波动 → 从 199 维压到 {n_90} 维。")

In [ ]:
# 6.2 看前 6 个主成分的"载荷"——每个主成分由哪些尺寸构成？
n_show = 6
components = pca.components_[:n_show]
loadings = pd.DataFrame(
    components.T, index=feature_cols,
    columns=[f'PC{i+1}' for i in range(n_show)])

# 每个主成分取 |载荷| 最大的 5 个尺寸
print("前 6 个主成分的核心构成尺寸（|载荷| Top 5）：")
for pc in loadings.columns:
    top = loadings[pc].abs().sort_values(ascending=False).head(5)
    print(f"\n【{pc}】（解释方差 {exp[int(pc[2:])-1]:.1%}）")
    for col, val in top.items():
        print(f"    {col:<14} 载载={loadings.loc[col, pc]:+.3f}")

In [ ]:
# 6.3 探索性因子分析（EFA）—— 更贴近"工艺因子"的建模
try:
    from factor_analyzer import FactorAnalyzer
    from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
    HAVE_FA = True
except ImportError:
    HAVE_FA = False
    print("⚠️ 未安装 factor_analyzer，跳过 EFA（可选：pip install factor_analyzer）")
    print("   下面仅用 PCA 结果演示。")

if HAVE_FA:
    # KMO 和 Bartlett 检验：判断数据是否适合做因子分析
    chi2, p_bart = calculate_bartlett_sphericity(df[feature_cols])
    kmo_all, kmo_model = calculate_kmo(df[feature_cols])
    print(f"Bartlett 球形检验: chi2={chi2:.1f}, p={p_bart:.3g}（p<0.05 → 适合做因子分析）")
    print(f"KMO = {kmo_model:.3f}（>0.6 适合，>0.8 良好）")

In [ ]:
# 6.4 拟合因子分析模型，提取 6 个因子（与真相对照）
if HAVE_FA:
    fa = FactorAnalyzer(n_factors=6, rotation='varimax')
    fa.fit(df[feature_cols])
    loadings_fa = pd.DataFrame(
        fa.loadings_, index=feature_cols,
        columns=[f'因子{i+1}' for i in range(6)])

    # 每个因子 Top 5 载荷尺寸
    print("6 个潜因子的核心构成尺寸（Varimax 旋转后 |载荷| Top 5）：")
    for f in loadings_fa.columns:
        top = loadings_fa[f].abs().sort_values(ascending=False).head(5)
        print(f"\n【{f}】")
        for col, val in top.items():
            print(f"    {col:<14} 载荷={loadings_fa.loc[col, f]:+.3f}")
    print("\n💡 对比第 0 章真相：F1 点胶/F2 屏幕贴合 是不良主因，因子分析能恢复出这个结构吗？")
else:
    # 退化方案：用 PCA 的前 6 个主成分得分代替
    scores_pca = pca.transform(X_scaled)[:, :6]
    print("用 PCA 前 6 个主成分得分代替因子得分：")
    print(pd.DataFrame(scores_pca, columns=[f'PC{i+1}' for i in range(6)]).describe())

In [ ]:
# 6.5 神奇对照：把还原出的因子与真实 F1~F6 比对（验证"挖对了"）
# 重新生成真实的 F（与第 0 章同样的种子，保证一致）
rng_check = np.random.default_rng(SEED)
F_true = rng_check.standard_normal((N, N_FACTORS))  # 这就是第0章的 F
# 真实 F1~F3 决定不良，F4~F6 无关
true_corrs = []
if HAVE_FA:
    scores_fa = fa.transform(df[feature_cols])  # 因子得分
    for i in range(6):
        row = []
        for j in range(6):
            row.append(np.corrcoef(scores_fa[:, i], F_true[:, j])[0, 1])
        true_corrs.append(row)
    corr_matrix = pd.DataFrame(true_corrs,
                               index=[f'还原因子{i+1}' for i in range(6)],
                               columns=[f'真实F{j+1}' for j in range(6)])
    plt.figure(figsize=(7, 5))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
    plt.title('还原因子 vs 真实因子 的相关性（|r| 高说明挖对了）')
    plt.tight_layout()
    plt.show()
    print("💡 解读：理想情况下每个还原因子只与一个真实因子强相关（对角线），说明成功还原。")
else:
    for i in range(6):
        for j in range(6):
            true_corrs.append(np.corrcoef(scores_pca[:, i], F_true[:, j])[0, 1])
    print("PCA 主成分与真实因子相关系数（最大者即对应）：")
    arr = np.array(true_corrs).reshape(6, 6)
    print(pd.DataFrame(arr, index=[f'PC{i+1}' for i in range(6)],
                      columns=[f'真实F{j+1}' for j in range(6)]).round(2))

### 📌 第 6 章小结

- **PCA / 因子分析** 把 199 个高度相关的尺寸压缩成 ~6 个**相互独立**的潜因子
- 每个因子背后是一组"共涨共跌"的尺寸，可赋予工艺含义（点胶、贴合、冲压…）
- 与第 0 章的生成真相对照，验证了分析方法的有效性
- **实战价值**：工程师不必盯 199 个 SPC 控制图，只需监控 6 个因子得分，大幅降负

---
## 第 7 章 · 机器学习对比：随机森林特征重要性

> 同一个"找关键尺寸"的问题，换一种完全不同的解法——**树模型**。
>
> 树模型的好处：
> - **天然处理共线性**（不像逻辑回归那么敏感）
> - **捕捉非线性关系**（Pearson 只看线性）
> - 直接输出**特征重要性**
>
> 我们把第 5 章（统计法）和本章（ML 法）的结果**并列对比**，这是教学重点。

In [ ]:
# 7.1 训练随机森林，看特征重要性
rf = RandomForestClassifier(n_estimators=200, max_depth=8,
                            random_state=42, class_weight='balanced')
rf.fit(df[feature_cols], df['label'])

rf_imp = pd.DataFrame({
    '尺寸': feature_cols,
    '重要性': rf.feature_importances_
}).sort_values('重要性', ascending=False).reset_index(drop=True)

print("随机森林特征重要性 Top 20：")
rf_imp.head(20)

In [ ]:
# 7.2 交叉验证 AUC（更客观地评估模型）
from sklearn.model_selection import cross_val_score
cv_auc = cross_val_score(rf, df[feature_cols], df['label'],
                         cv=5, scoring='roc_auc')

# 为了公平对比，多元逻辑回归也用相同的 5 折交叉验证
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
logit_pipe = make_pipeline(StandardScaler(),
                           LogisticRegression(max_iter=2000, class_weight='balanced'))
logit_cv_auc = cross_val_score(logit_pipe, df[feature_cols], df['label'],
                               cv=5, scoring='roc_auc')

print(f"随机森林 5折CV AUC:      {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")
print(f"多元逻辑回归 5折CV AUC:  {logit_cv_auc.mean():.3f} ± {logit_cv_auc.std():.3f}")
print()
print("💡 解读：两者都用 5 折交叉验证，可直接对比。")
print("   随机森林通常略高，因为它能捕捉非线性、对共线性也更稳健。")
print("   ⚠️ 注意：不要拿'全样本训练的 AUC'和'交叉验证 AUC'直接比——前者有乐观偏差。")

In [ ]:
# 7.3 三种方法大对比：逻辑回归 vs 随机森林 vs 点二列相关
# 把三种方法的"关键尺寸排名"放到一起看一致性
merge = risk[['尺寸', '点二列r', 'AUC']].rename(columns={'AUC': '单变量AUC'}).copy()
merge['单变量排名'] = merge['单变量AUC'].rank(ascending=False).astype(int)

# 多元逻辑回归里显著的尺寸
sig_cols = coef_df[coef_df['显著(p<0.05)']].index.tolist()
merge['多元显著'] = merge['尺寸'].isin(sig_cols)

# 随机森林重要性排名
merge = merge.merge(rf_imp[['尺寸', '重要性']].rename(columns={'重要性': 'RF重要性'}), on='尺寸', how='left')
merge['RF排名'] = merge['RF重要性'].rank(ascending=False).astype(int)
merge = merge.sort_values('单变量AUC', ascending=False).head(25)

print("🏆 三种方法综合视图（Top 25 尺寸）：")
print("列含义：单变量AUC | 是否多元显著 | RF重要性")
merge[['尺寸', '点二列r', '单变量AUC', '多元显著', 'RF重要性', 'RF排名']].to_string(index=False)

In [ ]:
# 7.4 可视化：三种方法排名的相关性（散点矩阵）
import matplotlib.pyplot as plt
plot_df = pd.DataFrame({
    '点二列|r|': pb_df.set_index('尺寸').loc[feature_cols, '|r|'].values,
    '单变量AUC': auc_df.set_index('尺寸').loc[feature_cols, 'AUC'].values,
    'RF重要性': rf_imp.set_index('尺寸').loc[feature_cols, '重要性'].values,
}, index=feature_cols)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
pairs_plot = [('点二列|r|', '单变量AUC'), ('点二列|r|', 'RF重要性'), ('单变量AUC', 'RF重要性')]
for ax, (a, b) in zip(axes, pairs_plot):
    ax.scatter(plot_df[a], plot_df[b], s=12, alpha=0.5)
    r = plot_df[[a, b]].corr().iloc[0, 1]
    ax.set_xlabel(a); ax.set_ylabel(b)
    ax.set_title(f'{a} vs {b}\nr = {r:.2f}')
plt.suptitle('三种"关键尺寸"评估方法的一致性', y=1.02)
plt.tight_layout()
plt.show()
print("💡 解读：三两两之间 r 越高，说明不同方法对'哪些尺寸重要'的判断越一致 → 结果可信。")

### 📌 第 7 章小结

| 维度 | 逻辑回归（第5章） | 随机森林（本章） |
|---|---|---|
| 处理共线性 | 敏感，需先 VIF 筛 | 天然稳健 |
| 非线性 | 只能线性 | 可捕捉 |
| 可解释性 | 系数/OR 值直接 | 只有重要性排序 |
| 假设 | 有分布假设 | 几乎无假设 |

**实战建议**：两种方法**都跑一遍**，结果一致的关键尺寸最可信；不一致的尺寸值得深挖（可能有交互效应）。

---
## 第 8 章 · 综合结论与实战练习题

### 8.1 全项目方法论地图

我们从一套 500×200 的手机产线尺寸数据出发，完整走过了**数据相关性分析的方法链条**：

```
数据生成/加载 (第0-1章)
    │
    ▼
EDA 探索 (第2章) ── 建立直觉
    │
    ├──► 连续×连续 相关 (第3章) ── Pearson/Spearman + 热力图 + 聚类
    │
    ├──► 0/1×连续 关联 (第4章) ⭐ ── 点二列/t检验/AUC 三件套 + FDR校正
    │         │
    │         ▼
    │    多元逻辑回归 (第5章) ── VIF 共线性诊断 + OR 值解读
    │
    ├──► 降维找因子 (第6章) ── PCA + 因子分析，还原 6 个潜因子
    │
    └──► ML 对比 (第7章) ── 随机森林，交叉验证
```

### 8.2 关键结论（对应本数据集）

1. **数据高度共线**：199 个尺寸背后其实是 ~6 个工艺因子在驱动（VIF 和 PCA 共同证实）
2. **不良主因**：F1（点胶密封）和 F2（屏幕贴合）两个潜因子，对应一批高载荷尺寸
3. **方法一致性**：点二列相关、多元逻辑回归、随机森林**三种方法指向同一批关键尺寸** → 可信度高
4. **可落地建议**：在组装前监控 F1/F2 相关的零件尺寸，可提前预警不良

### 8.3 🎯 实战练习题

把下面这些当作"换到真实产线数据时的检查清单"：

1. **数据生成**：把 `SEED` 改成别的值，重新生成数据。关键尺寸的排序会变吗？结论的**模式**（哪些因子是主因）会变吗？
2. **不良率调节**：把第 0 章的 `z = -2.6 + ...` 里的截距从 -2.6 改成 -1.0，不良率会升高。重跑分析，观察 AUC 变化（提示：不良样本越多，模型越好训）。
3. **引入非线性**：让某个尺寸与不良是**非线性**关系（如 `sin(F1)`），看 Pearson 和随机森林谁更能抓到。
4. **缺失值**：随机把 5% 的尺寸值置为 NaN，练习 `df.fillna` / `KNNImputer` 等处理。
5. **真实数据迁移**：如果你有产线真实 CSV，把第 1 章的 `df = pd.read_csv(...)` 替换掉，后续章节**几乎不用改**就能跑。注意检查：列名、缺失、量纲。
6. **可解释性进阶**：安装 `shap`（`pip install shap`），对第 7 章的随机森林算 SHAP 值，画 summary plot，看每个尺寸对不良的**方向性**贡献。
7. **时间维度**：如果数据带"生产时间戳"，试着做**时间序列分解**，看不良率是否有周期性（换班、换料批次）。

### 8.4 推荐延伸学习

- **偏相关分析**：控制其他变量后两变量的"净相关"
- **结构方程模型 (SEM)**：显式建模"潜因子 → 尺寸 → 不良"的因果链
- **DOE 试验设计**：主动设计实验找因果（而不只是观察性相关）
- **贝叶斯网络**：推断变量间的因果图

---

> ✅ **恭喜走完全程！** 下一步：把你的真实产线数据喂进来，这套方法几乎可以原样复用。
> 遇到问题记得回头看对应章节的"💡 教学提示"。